<a href="https://colab.research.google.com/github/lanaajs/Processamento-de-Linguagem-Natural-Python/blob/Aulas/Simplificador_Autom%C3%A1tico_de_Textos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###**Proposta: Simplificador Automático de Textos**

* **O Problema:** Alunos (ou pessoas em processo de alfabetização) encontram textos didáticos com vocabulário muito complexo e desistem de ler.

* **A Inovação:** Um "tradutor de complexidade". O usuário cola um texto difícil e o Transformer o reescreve usando apenas as 1.000 palavras mais comuns da língua portuguesa.

In [12]:
# instalando e importando bibliotecas
!pip install -q sentence-transformers pandas
import pandas as pd
from sentence_transformers import SentenceTransformer, util

In [13]:
# carregando o modelo
nome_do_modelo = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
modelo_bert = SentenceTransformer(nome_do_modelo)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
# dados
biblioteca_escolar = [
    "A fotossíntese é o processo de conversão de luz solar em energia química pelas plantas.",
    "O Ciclo da Água envolve evaporação, condensação e precipitação na atmosfera.",
    "A Primeira Lei de Newton afirma que um corpo em repouso tende a permanecer em repouso.",
    "Dom Casmurro é uma obra de Machado de Assis que explora o tema do ciúme e da dúvida."
]

In [15]:
# input do aluno (em linguagem simples/informal)
pesquisa_aluno = "como a planta produz alimento?"

In [16]:
# transforma tudo em vetores
embeddings_biblioteca = modelo_bert.encode(biblioteca_escolar, convert_to_tensor=True)
embedding_pesquisa = modelo_bert.encode(pesquisa_aluno, convert_to_tensor=True)

In [17]:
# calcula a similaridade entre a dúvida e os livros
scores = util.cos_sim(embedding_pesquisa, embeddings_biblioteca)[0]

In [20]:
# organização dos resultados
resultados = []

for i in range(len(biblioteca_escolar)):
    resultados.append({
        "Conteúdo do Livro": biblioteca_escolar[i],
        "Afinidade (Score)": f"{scores[i].item() * 100:.2f}%"
    })

pd.set_option('display.max_colwidth', None)
df_busca = pd.DataFrame(resultados).sort_values(by="Afinidade (Score)", ascending=False)

In [24]:
# imprime resultado
print(f"PESQUISA DO ALUNO: '{pesquisa_aluno}'")
print("-" * 50)
print(df_busca.to_string(index=False, justify='left'))
print("-" * 50)
print("Resultado: O Transformer identificou o conteúdo de 'Fotossíntese' como a melhor resposta.")

PESQUISA DO ALUNO: 'como a planta produz alimento?'
--------------------------------------------------
Conteúdo do Livro                                                                       Afinidade (Score)
A fotossíntese é o processo de conversão de luz solar em energia química pelas plantas. 44.10%           
 A Primeira Lei de Newton afirma que um corpo em repouso tende a permanecer em repouso.  4.44%           
           O Ciclo da Água envolve evaporação, condensação e precipitação na atmosfera. 22.83%           
   Dom Casmurro é uma obra de Machado de Assis que explora o tema do ciúme e da dúvida. -1.28%           
--------------------------------------------------
Resultado: O Transformer identificou o conteúdo de 'Fotossíntese' como a melhor resposta.
